In [4]:
import pandas as pd
import json

# Load data
orders = pd.read_csv('orders.csv')
with open('users.json', 'r') as f:
    users = pd.DataFrame(json.load(f))

# FIX: Convert all ID columns to string
orders['user_id'] = orders['user_id'].astype(str)
orders['restaurant_id'] = orders['restaurant_id'].astype(str)
users['user_id'] = users['user_id'].astype(str)

# Create simple restaurants DataFrame
restaurants = pd.DataFrame({
    'restaurant_id': orders['restaurant_id'].unique(),
    'cuisine_type': ['Cuisine_' + str(i) for i in range(len(orders['restaurant_id'].unique()))]
})
restaurants['restaurant_id'] = restaurants['restaurant_id'].astype(str)

# Merge
merged = pd.merge(orders, users, on='user_id', how='left')
final_df = pd.merge(merged, restaurants, on='restaurant_id', how='left')

# Save
final_df.to_csv('final_food_delivery_dataset.csv', index=False)
print("Dataset created successfully!")

Dataset created successfully!


In [3]:
import pandas as pd
import json

# Load data
orders_df = pd.read_csv('orders.csv')
with open('users.json', 'r') as f:
    users_df = pd.DataFrame(json.load(f))

# Merge the data
merged_df = pd.merge(orders_df, users_df, on='user_id', how='left')

# Filter for Gold members only
gold_members_df = merged_df[merged_df['membership'] == 'Gold']

# Group by city and calculate total revenue
city_revenue = gold_members_df.groupby('city')['total_amount'].sum()

# Sort to find highest
city_revenue_sorted = city_revenue.sort_values(ascending=False)

print("Top 5 cities by Gold member revenue:")
print(city_revenue_sorted.head())

# Get the answer
highest_revenue_city = city_revenue_sorted.index[0]
highest_revenue_amount = city_revenue_sorted.iloc[0]

print(f"\n City with highest Gold member revenue: {highest_revenue_city}")
print(f"   Total revenue from Gold members: ${highest_revenue_amount:,.2f}")


Top 5 cities by Gold member revenue:
city
Chennai      1080909.79
Pune         1003012.32
Bangalore     994702.59
Hyderabad     896740.19
Name: total_amount, dtype: float64

 City with highest Gold member revenue: Chennai
   Total revenue from Gold members: $1,080,909.79


In [7]:
import pandas as pd
import json
import re

# ============================================
# STEP 1: LOAD ORDERS AND USERS
# ============================================
print("Loading orders.csv and users.json...")
orders_df = pd.read_csv('orders.csv')
with open('users.json', 'r') as f:
    users_df = pd.DataFrame(json.load(f))

# Convert IDs to string
orders_df['user_id'] = orders_df['user_id'].astype(str)
orders_df['restaurant_id'] = orders_df['restaurant_id'].astype(str)
users_df['user_id'] = users_df['user_id'].astype(str)

# ============================================
# STEP 2: LOAD RESTAURANTS.SQL WITH REAL DATA
# ============================================
print("\nLoading restaurants.sql...")

# Read the SQL file
with open('restaurants.sql', 'r') as f:
    sql_content = f.read()

# METHOD 1: Check if it's a simple CSV-like format
print("Analyzing file format...")

# Look for INSERT statements with real data
# Common pattern: INSERT INTO restaurants VALUES (1, 'Restaurant Name', 'Cuisine Type', ...);
pattern = r"INSERT INTO.*?VALUES\s*\((.*?)\);"

matches = re.findall(pattern, sql_content, re.DOTALL | re.IGNORECASE)

if matches:
    print(f"Found {len(matches)} INSERT statements")
    
    # Parse all rows
    all_rows = []
    for match in matches:
        # Split by comma, but be careful with commas inside quotes
        values = []
        current = ""
        in_quotes = False
        
        for char in match:
            if char == "'" or char == '"':
                in_quotes = not in_quotes
                current += char
            elif char == ',' and not in_quotes:
                values.append(current.strip())
                current = ""
            else:
                current += char
        if current:
            values.append(current.strip())
        
        # Clean values (remove quotes)
        cleaned_values = []
        for v in values:
            v = v.strip()
            if (v.startswith("'") and v.endswith("'")) or (v.startswith('"') and v.endswith('"')):
                v = v[1:-1]
            cleaned_values.append(v)
        
        all_rows.append(cleaned_values)
    
    # Create DataFrame
    # Check how many columns we have
    num_cols = len(all_rows[0]) if all_rows else 0
    print(f"Each row has {num_cols} columns")
    
    # Common column names for restaurants table
    column_names = ['restaurant_id', 'name', 'cuisine', 'city', 'address', 'phone', 'rating']
    
    # Use available columns (may have fewer than 6)
    available_columns = column_names[:num_cols]
    
    restaurants_df = pd.DataFrame(all_rows, columns=available_columns)
    
    print(f"Restaurants loaded: {len(restaurants_df)} rows")
    print(f"Columns: {restaurants_df.columns.tolist()}")
    
    # Show sample cuisine values
    if 'cuisine' in restaurants_df.columns:
        print("\nSample cuisine names found:")
        print(restaurants_df['cuisine'].unique()[:10])
    
else:
    # METHOD 2: Check if file is actually CSV/TSV format
    print("No INSERT statements found. Checking if it's CSV/TSV...")
    
    # Read first few lines to see format
    with open('restaurants.sql', 'r') as f:
        first_lines = [next(f) for _ in range(10)]
    
    print("First 10 lines of restaurants.sql:")
    for i, line in enumerate(first_lines):
        print(f"{i}: {line.rstrip()}")
    
    # Try to read as CSV if it looks like data
    if ',' in first_lines[0] and len(first_lines[0].split(',')) > 1:
        restaurants_df = pd.read_csv('restaurants.sql')
        print("Successfully read as CSV!")
    else:
        print("\nERROR: Could not parse restaurants.sql properly!")
        print("Please check the file format and share what you see.")
        exit()

# ============================================
# STEP 3: MERGE ALL DATA
# ============================================
print("\nMerging data...")

# Make sure restaurant_id is string for merging
restaurants_df['restaurant_id'] = restaurants_df['restaurant_id'].astype(str)

# Merge orders with restaurants FIRST to get real cuisine names
merged_orders_restaurants = pd.merge(
    orders_df,
    restaurants_df[['restaurant_id', 'cuisine']],  # Get only needed columns
    on='restaurant_id',
    how='left'
)

# Then merge with users
final_df = pd.merge(
    merged_orders_restaurants,
    users_df,
    on='user_id',
    how='left'
)

print(f"Final dataset shape: {final_df.shape}")
print(f"Columns: {final_df.columns.tolist()}")

# ============================================
# STEP 4: FIND CUISINE WITH HIGHEST AVERAGE ORDER VALUE
# ============================================
print("\n" + "="*60)
print("ANALYZING CUISINE PERFORMANCE")
print("="*60)

# Check if we have real cuisine names
if 'cuisine' in final_df.columns:
    # Check what cuisine values we have
    unique_cuisines = final_df['cuisine'].unique()
    print(f"\nFound {len(unique_cuisines)} unique cuisine types:")
    print(unique_cuisines[:20])  # Show first 20
    
    # Find amount column
    if 'total_amount' in final_df.columns:
        amount_col = 'total_amount'
    elif 'order_value' in final_df.columns:
        amount_col = 'order_value'
    else:
        # Try to find amount column
        amount_cols = [col for col in final_df.columns if 'amount' in col.lower() or 'value' in col.lower()]
        if amount_cols:
            amount_col = amount_cols[0]
        else:
            print("ERROR: No amount column found!")
            print("Available columns:", final_df.columns.tolist())
            amount_col = None
    
    if amount_col:
        # Calculate average order value by cuisine
        cuisine_avg = final_df.groupby('cuisine')[amount_col].mean()
        
        # Sort descending
        cuisine_avg_sorted = cuisine_avg.sort_values(ascending=False)
        
        print(f"\n{'='*60}")
        print("AVERAGE ORDER VALUE BY CUISINE:")
        print('='*60)
        
        # Display top 10
        for i, (cuisine, avg_value) in enumerate(cuisine_avg_sorted.head(10).items(), 1):
            print(f"{i:2}. {cuisine:20} ${avg_value:.2f}")
        
        
        
        # Save to CSV for reference
        final_df.to_csv('final_food_delivery_dataset_with_real_cuisines.csv', index=False)
       
        
else:
    print("ERROR: 'cuisine' column not found in final dataset!")
    print("Available columns:", final_df.columns.tolist())

Loading orders.csv and users.json...

Loading restaurants.sql...
Analyzing file format...
Found 500 INSERT statements
Each row has 4 columns
Restaurants loaded: 500 rows
Columns: ['restaurant_id', 'name', 'cuisine', 'city']

Sample cuisine names found:
<StringArray>
['Chinese', 'Indian', 'Mexican', 'Italian']
Length: 4, dtype: str

Merging data...
Final dataset shape: (10000, 10)
Columns: ['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount', 'restaurant_name', 'cuisine', 'name', 'city', 'membership']

ANALYZING CUISINE PERFORMANCE

Found 4 unique cuisine types:
<StringArray>
['Mexican', 'Indian', 'Chinese', 'Italian']
Length: 4, dtype: str

AVERAGE ORDER VALUE BY CUISINE:
 1. Mexican              $808.02
 2. Italian              $799.45
 3. Indian               $798.47
 4. Chinese              $798.39


In [13]:
import pandas as pd
import json
import re

# ============================================
# STEP 1: LOAD ALL DATA
# ============================================
print("Loading data...")

# Load orders
orders_df = pd.read_csv('orders.csv')

# Load users
with open('users.json', 'r') as f:
    users_df = pd.DataFrame(json.load(f))

# Convert IDs to string
orders_df['user_id'] = orders_df['user_id'].astype(str)
orders_df['restaurant_id'] = orders_df['restaurant_id'].astype(str)
users_df['user_id'] = users_df['user_id'].astype(str)

# ============================================
# STEP 2: CALCULATE TOTAL ORDER VALUE PER USER
# ============================================
print("\nCalculating total order value per user...")

# Find the correct amount column name
if 'total_amount' in orders_df.columns:
    amount_col = 'total_amount'
elif 'order_value' in orders_df.columns:
    amount_col = 'order_value'
else:
    # Find any amount column
    amount_cols = [col for col in orders_df.columns if 'amount' in col.lower() or 'value' in col.lower()]
    if amount_cols:
        amount_col = amount_cols[0]
    else:
        print("ERROR: Could not find amount column!")
        print("Available columns in orders:", orders_df.columns.tolist())
        exit()

print(f"Using '{amount_col}' as amount column")

# Calculate total amount spent by each user
user_total_spent = orders_df.groupby('user_id')[amount_col].sum().reset_index()
user_total_spent.columns = ['user_id', 'total_spent']

print(f"Total users found: {len(user_total_spent)}")

# ============================================
# STEP 3: COUNT USERS WITH TOTAL > ₹1000
# ============================================
print("\n" + "="*60)
print("ANALYSIS: Users with total orders > ₹1000")
print("="*60)

# Count users who spent more than ₹1000
users_over_1000 = user_total_spent[user_total_spent['total_spent'] > 1000]
count_over_1000 = len(users_over_1000)

print(f"Number of distinct users with total orders > ₹1000: {count_over_1000}")

# Show some statistics
print(f"\nStatistics of total spending per user:")
print(f"Minimum: ₹{user_total_spent['total_spent'].min():.2f}")
print(f"Maximum: ₹{user_total_spent['total_spent'].max():.2f}")
print(f"Average: ₹{user_total_spent['total_spent'].mean():.2f}")
print(f"Median: ₹{user_total_spent['total_spent'].median():.2f}")

# ============================================
# STEP 4: FIND THE CORRECT RANGE
# ============================================
print("\n" + "="*60)
print("DETERMINING THE CORRECT RANGE")
print("="*60)

# Check which range the count falls into
if count_over_1000 < 500:
    answer_range = "< 500"
    answer_explanation = f"Count ({count_over_1000}) is less than 500"
elif 500 <= count_over_1000 <= 1000:
    answer_range = "500 – 1000"
    answer_explanation = f"Count ({count_over_1000}) is between 500 and 1000 inclusive"
elif 1001 <= count_over_1000 <= 2000:
    answer_range = "1000 – 2000"
    answer_explanation = f"Count ({count_over_1000}) is between 1001 and 2000"
else:  # count_over_1000 > 2000
    answer_range = "> 2000"
    answer_explanation = f"Count ({count_over_1000}) is greater than 2000"

print(f"\nResult: {count_over_1000} users have total orders > ₹1000")

Loading data...

Calculating total order value per user...
Using 'total_amount' as amount column
Total users found: 2883

ANALYSIS: Users with total orders > ₹1000
Number of distinct users with total orders > ₹1000: 2544

Statistics of total spending per user:
Minimum: ₹102.22
Maximum: ₹11556.49
Average: ₹2778.92
Median: ₹2514.92

DETERMINING THE CORRECT RANGE

Result: 2544 users have total orders > ₹1000


In [4]:
import pandas as pd
import json
import re

# ============================================
# STEP 1: LOAD AND PARSE RESTAURANTS DATA
# ============================================
print("Loading restaurants.sql...")

with open('restaurants.sql', 'r') as f:
    sql_content = f.read()

# Parse INSERT statements
restaurant_data = []
pattern = r"INSERT INTO.*?VALUES\s*\((.*?)\);"
matches = re.findall(pattern, sql_content, re.DOTALL | re.IGNORECASE)

if matches:
    for match in matches:
        parts = [p.strip().strip("'").strip('"') for p in match.split(',')]
        restaurant_data.append(parts)
    
    # Create DataFrame with correct columns
    # Based on your data: restaurant_id, name, cuisine, rating
    restaurants_df = pd.DataFrame(restaurant_data, columns=['restaurant_id', 'name', 'cuisine', 'rating'])
    
    # Convert rating to numeric (it's in the "city" position but actually contains ratings)
    restaurants_df['rating'] = pd.to_numeric(restaurants_df['rating'], errors='coerce')
    
    print(f"Loaded {len(restaurants_df)} restaurants")
    print("\nSample restaurants with ratings:")
    print(restaurants_df.head())
    
    print(f"\nRating statistics:")
    print(f"Min rating: {restaurants_df['rating'].min():.1f}")
    print(f"Max rating: {restaurants_df['rating'].max():.1f}")
    print(f"Avg rating: {restaurants_df['rating'].mean():.1f}")

# ============================================
# STEP 2: LOAD ORDERS AND MERGE
# ============================================
print("\nLoading orders and merging...")
orders_df = pd.read_csv('orders.csv')

# Convert IDs to string
orders_df['restaurant_id'] = orders_df['restaurant_id'].astype(str)
restaurants_df['restaurant_id'] = restaurants_df['restaurant_id'].astype(str)

# Merge orders with restaurant ratings
merged_df = pd.merge(orders_df, restaurants_df[['restaurant_id', 'rating']], 
                     on='restaurant_id', how='left')

print(f"Total orders: {len(merged_df)}")
print(f"Orders with rating info: {merged_df['rating'].notna().sum()}")

# ============================================
# STEP 3: CATEGORIZE RATINGS INTO RANGES
# ============================================
print("\nCategorizing ratings into ranges...")

def categorize_rating(rating):
    if pd.isna(rating):
        return 'No Rating'
    elif 3.0 <= rating <= 3.5:
        return '3.0 – 3.5'
    elif 3.6 <= rating <= 4.0:
        return '3.6 – 4.0'
    elif 4.1 <= rating <= 4.5:
        return '4.1 – 4.5'
    elif 4.6 <= rating <= 5.0:
        return '4.6 – 5.0'
    else:
        return f'Other ({rating})'

merged_df['rating_range'] = merged_df['rating'].apply(categorize_rating)

# ============================================
# STEP 4: CALCULATE REVENUE BY RATING RANGE
# ============================================
print("\nCalculating revenue by rating range...")

# Find amount column
if 'total_amount' in merged_df.columns:
    amount_col = 'total_amount'
elif 'order_value' in merged_df.columns:
    amount_col = 'order_value'
else:
    amount_cols = [col for col in merged_df.columns if 'amount' in col.lower() or 'value' in col.lower()]
    amount_col = amount_cols[0] if amount_cols else None

print(f"Using '{amount_col}' as amount column")

# Calculate total revenue by rating range
revenue_by_range = merged_df.groupby('rating_range')[amount_col].sum().sort_values(ascending=False)

# Filter only the ranges in the question
valid_ranges = ['3.0 – 3.5', '3.6 – 4.0', '4.1 – 4.5', '4.6 – 5.0']
revenue_by_range_filtered = revenue_by_range[revenue_by_range.index.isin(valid_ranges)]

# ============================================
# STEP 5: FIND THE ANSWER
# ============================================
print("\n" + "="*60)
print("REVENUE BY RESTAURANT RATING RANGE")
print("="*60)

print("\nAll rating ranges:")
for i, (rating_range, revenue) in enumerate(revenue_by_range.head().items(), 1):
    print(f"{i}. {rating_range}: ₹{revenue:,.2f}")

print("\n" + "-"*40)
print("Ranges from the multiple choice question:")
for i, (rating_range, revenue) in enumerate(revenue_by_range_filtered.items(), 1):
    print(f"{i}. {rating_range}: ₹{revenue:,.2f}")

# Get the highest revenue range
if not revenue_by_range_filtered.empty:
    highest_range = revenue_by_range_filtered.index[0]
    highest_revenue = revenue_by_range_filtered.iloc[0]
    
    print(f"\nANSWER: {highest_range}")
    print(f"   Generated ₹{highest_revenue:,.2f} in revenue")
else:
    print("\n⚠️ WARNING: No data found for the specified rating ranges!")
    print("Check if any restaurants have ratings in these ranges.")



Loading restaurants.sql...
Loaded 500 restaurants

Sample restaurants with ratings:
  restaurant_id          name  cuisine  rating
0             1  Restaurant_1  Chinese     4.8
1             2  Restaurant_2   Indian     4.1
2             3  Restaurant_3  Mexican     4.3
3             4  Restaurant_4  Chinese     4.1
4             5  Restaurant_5  Chinese     4.8

Rating statistics:
Min rating: 3.0
Max rating: 5.0
Avg rating: 4.0

Loading orders and merging...
Total orders: 10000
Orders with rating info: 10000

Categorizing ratings into ranges...

Calculating revenue by rating range...
Using 'total_amount' as amount column

REVENUE BY RESTAURANT RATING RANGE

All rating ranges:
1. 4.6 – 5.0: ₹2,197,030.75
2. 3.0 – 3.5: ₹2,136,772.70
3. 4.1 – 4.5: ₹1,960,326.26
4. 3.6 – 4.0: ₹1,717,494.41

----------------------------------------
Ranges from the multiple choice question:
1. 4.6 – 5.0: ₹2,197,030.75
2. 3.0 – 3.5: ₹2,136,772.70
3. 4.1 – 4.5: ₹1,960,326.26
4. 3.6 – 4.0: ₹1,717,494.41

ANSW

In [8]:
import pandas as pd
import json

# Load and merge data
orders = pd.read_csv('orders.csv')
with open('users.json', 'r') as f:
    users = pd.DataFrame(json.load(f))

# Merge
merged = pd.merge(orders, users, on='user_id', how='left')

# Filter Gold members
if 'membership' in merged.columns:
    gold_df = merged[merged['membership'] == 'Gold']
else:
    member_col = [col for col in merged.columns if 'member' in col.lower()][0]
    gold_df = merged[merged[member_col] == 'Gold']

# Find columns
city_col = 'city' if 'city' in gold_df.columns else [col for col in gold_df.columns if 'city' in col.lower()][0]
amount_col = 'total_amount' if 'total_amount' in gold_df.columns else 'order_value'

# Calculate average by city
avg_by_city = gold_df.groupby(city_col)[amount_col].mean()

# Check against options
options = ['Hyderabad', 'Bangalore', 'Chennai', 'Pune']

# Find which option exists in data
for option in options:
    if option in avg_by_city.index:
        print(f"Found {option}: ₹{avg_by_city[option]:.2f}")

# Get highest among options that exist
existing_options = {city: avg_by_city[city] for city in options if city in avg_by_city.index}

if existing_options:
    answer = max(existing_options, key=existing_options.get)
    print(f"\n ANSWER: {answer} (₹{existing_options[answer]:.2f})")
else:
    print("None of the option cities found in data!")
    print("Cities in data:", avg_by_city.index.tolist())

Found Hyderabad: ₹806.42
Found Bangalore: ₹793.22
Found Chennai: ₹808.46
Found Pune: ₹781.16

 ANSWER: Chennai (₹808.46)


In [11]:
import pandas as pd
import json
import re

# Load restaurants to get cuisine data
with open('restaurants.sql', 'r') as f:
    sql_content = f.read()

# Parse restaurants
data = []
pattern = r"INSERT INTO.*?VALUES\s*\((.*?)\);"
matches = re.findall(pattern, sql_content, re.DOTALL | re.IGNORECASE)

for match in matches:
    parts = [p.strip().strip("'").strip('"') for p in match.split(',')]
    data.append(parts)

# Create restaurants DataFrame (restaurant_id, name, cuisine, rating)
restaurants = pd.DataFrame(data, columns=['restaurant_id', 'name', 'cuisine', 'rating'])
restaurants['restaurant_id'] = restaurants['restaurant_id'].astype(str)

# Load orders
orders = pd.read_csv('orders.csv')
orders['restaurant_id'] = orders['restaurant_id'].astype(str)

# Merge to get cuisine for each order
merged = pd.merge(orders, restaurants[['restaurant_id', 'cuisine']], 
                  on='restaurant_id', how='left')

# Find amount column
amount_col = 'total_amount' if 'total_amount' in merged.columns else 'order_value'

# Calculate metrics for each cuisine
cuisine_stats = merged.groupby('cuisine').agg({
    'restaurant_id': 'nunique',  # distinct restaurants
    amount_col: 'sum'            # total revenue
}).rename(columns={'restaurant_id': 'num_restaurants', amount_col: 'total_revenue'})

# Sort by number of restaurants (ascending) to find cuisines with fewest restaurants
cuisine_stats = cuisine_stats.sort_values('num_restaurants')

print("\nCuisines sorted by number of distinct restaurants (fewest first):")
print(cuisine_stats)

# Filter only the cuisines in the options
options = ['Indian', 'Chinese', 'Italian', 'Mexican']
cuisine_stats_filtered = cuisine_stats[cuisine_stats.index.isin(options)]

print("\n" + "="*60)
print("ANALYSIS FOR OPTION CUISINES:")
print("="*60)

for cuisine in options:
    if cuisine in cuisine_stats.index:
        stats = cuisine_stats.loc[cuisine]
        print(f"\n{cuisine}:")
        print(f"  Distinct restaurants: {stats['num_restaurants']}")
        print(f"  Total revenue: ₹{stats['total_revenue']:,.2f}")
        if stats['num_restaurants'] > 0:
            print(f"  Revenue per restaurant: ₹{stats['total_revenue']/stats['num_restaurants']:,.2f}")

# Find which has lowest number of restaurants
lowest_restaurants_cuisine = cuisine_stats_filtered['num_restaurants'].idxmin()
lowest_count = cuisine_stats_filtered['num_restaurants'].min()

print(f"\nCUISINE WITH FEWEST RESTAURANTS: {lowest_restaurants_cuisine} ({lowest_count} restaurants)")

# Check if it still has significant revenue
significant_threshold = cuisine_stats_filtered['total_revenue'].median()  # median revenue as threshold
lowest_revenue = cuisine_stats_filtered.loc[lowest_restaurants_cuisine, 'total_revenue']

print(f"   Its revenue: ₹{lowest_revenue:,.2f}")
print(f"   Median revenue threshold: ₹{significant_threshold:,.2f}")

if lowest_revenue >= significant_threshold * 0.5:  # At least 50% of median
    print(f" This cuisine still contributes significant revenue!")
    answer = lowest_restaurants_cuisine
else:
    # Find next cuisine with few restaurants but good revenue
    print("\nLooking for alternative...")
    sorted_by_restaurants = cuisine_stats_filtered.sort_values('num_restaurants')
    for cuisine in sorted_by_restaurants.index:
        revenue = sorted_by_restaurants.loc[cuisine, 'total_revenue']
        if revenue >= significant_threshold * 0.5:
            answer = cuisine
            print(f"{cuisine} has few restaurants ({sorted_by_restaurants.loc[cuisine, 'num_restaurants']}) and good revenue (₹{revenue:,.2f})")
            break

print(f"\nFINAL ANSWER: {answer}")


Cuisines sorted by number of distinct restaurants (fewest first):
         num_restaurants  total_revenue
cuisine                                
Chinese              120     1930504.65
Indian               126     1971412.58
Italian              126     2024203.80
Mexican              128     2085503.09

ANALYSIS FOR OPTION CUISINES:

Indian:
  Distinct restaurants: 126.0
  Total revenue: ₹1,971,412.58
  Revenue per restaurant: ₹15,646.13

Chinese:
  Distinct restaurants: 120.0
  Total revenue: ₹1,930,504.65
  Revenue per restaurant: ₹16,087.54

Italian:
  Distinct restaurants: 126.0
  Total revenue: ₹2,024,203.80
  Revenue per restaurant: ₹16,065.11

Mexican:
  Distinct restaurants: 128.0
  Total revenue: ₹2,085,503.09
  Revenue per restaurant: ₹16,292.99

CUISINE WITH FEWEST RESTAURANTS: Chinese (120 restaurants)
   Its revenue: ₹1,930,504.65
   Median revenue threshold: ₹1,997,808.19
 This cuisine still contributes significant revenue!

FINAL ANSWER: Chinese


In [13]:
import pandas as pd
import json

# Load and merge data
orders = pd.read_csv('orders.csv')
with open('users.json', 'r') as f:
    users = pd.DataFrame(json.load(f))

merged = pd.merge(orders, users, on='user_id', how='left')

# Find membership column
if 'membership' in merged.columns:
    member_col = 'membership'
else:
    member_col = [col for col in merged.columns if 'member' in col.lower()][0]

# Calculate percentage
total_orders = len(merged)
gold_orders = len(merged[merged[member_col] == 'Gold'])
percentage = (gold_orders / total_orders) * 100
rounded_percentage = round(percentage)

print(f"Total orders: {total_orders}")
print(f"Gold member orders: {gold_orders}")
print(f"Percentage: {percentage:.2f}%")
print(f"Rounded: {rounded_percentage}%")

# Match with options
options = [40, 45, 50, 55]
closest_option = min(options, key=lambda x: abs(x - rounded_percentage))

print(f"\nANSWER: {closest_option}%")
print(f"   (Closest option to {rounded_percentage}%)")

Total orders: 10000
Gold member orders: 4987
Percentage: 49.87%
Rounded: 50%

ANSWER: 50%
   (Closest option to 50%)


In [15]:
import pandas as pd

# ============================================
# STEP 1: LOAD YOUR MERGED DATASET
# ============================================
print("Loading merged dataset...")
df = pd.read_csv('final_food_delivery_dataset.csv')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# ============================================
# STEP 2: IDENTIFY COLUMN NAMES
# ============================================
print("\nIdentifying column names...")

# Find restaurant name column
name_cols = [col for col in df.columns if 'name' in col.lower() and 'restaurant' in col.lower()]
if name_cols:
    restaurant_name_col = name_cols[0]
else:
    # Try other patterns
    name_cols = [col for col in df.columns if 'name' in col.lower()]
    if name_cols:
        restaurant_name_col = name_cols[0]
    else:
        # Last resort: check all columns
        print("Available columns:")
        for col in df.columns:
            print(f"  - {col}")
        restaurant_name_col = input("Enter the restaurant name column: ")

print(f"Using '{restaurant_name_col}' as restaurant name column")

# Find order ID column (for counting)
order_cols = [col for col in df.columns if 'order' in col.lower() and 'id' in col.lower()]
if order_cols:
    order_id_col = order_cols[0]
else:
    order_id_col = 'order_id' if 'order_id' in df.columns else df.columns[0]

print(f"Using '{order_id_col}' as order ID column")

# Find amount column
if 'total_amount' in df.columns:
    amount_col = 'total_amount'
elif 'order_value' in df.columns:
    amount_col = 'order_value'
else:
    amount_cols = [col for col in df.columns if 'amount' in col.lower() or 'value' in col.lower()]
    if amount_cols:
        amount_col = amount_cols[0]
    else:
        print("ERROR: No amount column found!")
        exit()

print(f"Using '{amount_col}' as amount column")

# ============================================
# STEP 3: CHECK RESTAURANT NAMES IN DATA
# ============================================
print("\n" + "="*60)
print("CHECKING RESTAURANT NAMES IN DATASET")
print("="*60)

# Get unique restaurant names
unique_restaurants = df[restaurant_name_col].unique()
print(f"Total unique restaurants: {len(unique_restaurants)}")
print("\nFirst 20 restaurant names:")
for i, name in enumerate(unique_restaurants[:20], 1):
    print(f"{i:2}. {name}")

# Check for the option restaurants
options = ['Grand Cafe Punjabi', 'Grand Restaurant South Indian', 
           'Ruchi Mess Multicuisine', 'Ruchi Foods Chinese']

print(f"\nLooking for option restaurants:")
found_options = []
for option in options:
    # Check for exact match
    exact_matches = df[df[restaurant_name_col] == option]
    
    # Check for partial match (case-insensitive)
    partial_matches = df[df[restaurant_name_col].str.contains(option, case=False, na=False)]
    
    if not exact_matches.empty:
        print(f"✓ Exact match found: '{option}'")
        found_options.append(option)
    elif not partial_matches.empty:
        actual_name = partial_matches[restaurant_name_col].iloc[0]
        print(f"✓ Partial match: '{actual_name}' contains '{option}'")
        found_options.append(actual_name)
    else:
        print(f"✗ Not found: '{option}'")

# ============================================
# STEP 4: CALCULATE RESTAURANT STATISTICS
# ============================================
print("\n" + "="*60)
print("CALCULATING RESTAURANT STATISTICS")
print("="*60)

# Calculate for ALL restaurants first
all_restaurant_stats = df.groupby(restaurant_name_col).agg(
    total_orders=(order_id_col, 'count'),
    avg_order_value=(amount_col, 'mean'),
    total_revenue=(amount_col, 'sum')
).reset_index()

print(f"\nCalculated stats for {len(all_restaurant_stats)} restaurants")

# ============================================
# STEP 5: FILTER FOR RESTAURANTS WITH <20 ORDERS
# ============================================
print("\n" + "="*60)
print("FILTERING: RESTAURANTS WITH LESS THAN 20 ORDERS")
print("="*60)

# Filter restaurants with less than 20 total orders
filtered_restaurants = all_restaurant_stats[all_restaurant_stats['total_orders'] < 20]

print(f"Restaurants with <20 orders: {len(filtered_restaurants)}")
print(f"Restaurants with ≥20 orders: {len(all_restaurant_stats) - len(filtered_restaurants)}")

if len(filtered_restaurants) == 0:
    print("⚠️ No restaurants with less than 20 orders found!")
    print("Try adjusting the threshold or check your data.")
    exit()

# Sort by average order value (descending)
filtered_restaurants = filtered_restaurants.sort_values('avg_order_value', ascending=False)

print("\nTop 10 restaurants with <20 orders (highest average order value):")
print(filtered_restaurants[['total_orders', 'avg_order_value', 'total_revenue', restaurant_name_col]].head(10))

# ============================================
# STEP 6: FIND THE ANSWER FROM OPTIONS
# ============================================
print("\n" + "="*60)
print("FINDING ANSWER FROM OPTIONS")
print("="*60)

# Check which option restaurants exist in our filtered list
answer_candidates = []

for option in options:
    # Check for exact or partial match in filtered restaurants
    matches = filtered_restaurants[
        filtered_restaurants[restaurant_name_col].str.contains(option, case=False, na=False)
    ]
    
    if not matches.empty:
        for _, row in matches.iterrows():
            answer_candidates.append({
                'name': row[restaurant_name_col],
                'option_name': option,
                'total_orders': row['total_orders'],
                'avg_order_value': row['avg_order_value'],
                'total_revenue': row['total_revenue']
            })

if answer_candidates:
    print(f"\nFound {len(answer_candidates)} candidate restaurants from options:")
    
    # Sort candidates by average order value
    answer_candidates.sort(key=lambda x: x['avg_order_value'], reverse=True)
    
    for i, candidate in enumerate(answer_candidates, 1):
        print(f"\n{i}. {candidate['name']} (matches '{candidate['option_name']}'):")
        print(f"   Orders: {candidate['total_orders']}")
        print(f"   Avg Order Value: ₹{candidate['avg_order_value']:.2f}")
        print(f"   Total Revenue: ₹{candidate['total_revenue']:.2f}")
    
    # The answer is the one with highest average order value
    best_candidate = answer_candidates[0]
    
    print(f"\n" + "="*60)
    print(f"📝 FINAL ANSWER: {best_candidate['option_name']}")
    print(f"   (Matching restaurant: {best_candidate['name']})")
    print(f"   Orders: {best_candidate['total_orders']}")
    print(f"   Average Order Value: ₹{best_candidate['avg_order_value']:.2f}")
    print("="*60)
    
else:
    # If no option restaurants found in filtered list, check all restaurants
    print("\n⚠️ No option restaurants found with <20 orders!")
    print("\nChecking all restaurants (not filtered by <20 orders):")
    
    for option in options:
        matches = all_restaurant_stats[
            all_restaurant_stats[restaurant_name_col].str.contains(option, case=False, na=False)
        ]
        
        if not matches.empty:
            print(f"\nFound '{option}' but with {matches.iloc[0]['total_orders']} orders:")
            print(matches[[restaurant_name_col, 'total_orders', 'avg_order_value']])
    
    # Alternative: Find the restaurant with highest average among ALL with <20 orders
    if not filtered_restaurants.empty:
        best_overall = filtered_restaurants.iloc[0]
        print(f"\n" + "="*60)
        print("ALTERNATIVE: Highest average among ALL restaurants with <20 orders:")
        print(f"Restaurant: {best_overall[restaurant_name_col]}")
        print(f"Orders: {best_overall['total_orders']}")
        print(f"Avg Order Value: ₹{best_overall['avg_order_value']:.2f}")
        print("="*60)
        
        # Check if this restaurant name contains any of the options
        for option in options:
            if option.lower() in best_overall[restaurant_name_col].lower():
                print(f"\n📝 THIS MATCHES OPTION: {option}")
                break



Loading merged dataset...
Dataset shape: (10000, 10)
Columns: ['order_id', 'user_id', 'restaurant_id', 'order_date', 'total_amount', 'restaurant_name', 'name', 'city', 'membership', 'cuisine_type']

Identifying column names...
Using 'restaurant_name' as restaurant name column
Using 'order_id' as order ID column
Using 'total_amount' as amount column

CHECKING RESTAURANT NAMES IN DATASET
Total unique restaurants: 433

First 20 restaurant names:
 1. New Foods Chinese
 2. Ruchi Curry House Multicuisine
 3. Spice Kitchen Punjabi
 4. Darbar Kitchen Non-Veg
 5. Royal Eatery South Indian
 6. Annapurna Tiffins South Indian
 7. Royal Biryani North Indian
 8. Spice Mess Punjabi
 9. Ruchi Biryani Punjabi
10. Taste of Biryani Non-Veg
11. Amma Delights Family Restaurant
12. Royal Tiffins Multicuisine
13. Amma Tiffins South Indian
14. Grand Cafe Punjabi
15. Amma Biryani North Indian
16. Amma Restaurant South Indian
17. Ruchi Foods Chinese
18. Darbar Delights South Indian
19. Spice Mess Andhra
20. Udu

In [18]:
import pandas as pd
import re
import json

# ============================================
# STEP 1: LOAD RESTAURANTS.SQL WITH ACTUAL CUISINE NAMES
# ============================================
print("Loading restaurants.sql to get real cuisine names...")

with open('restaurants.sql', 'r') as f:
    sql_content = f.read()

# Parse the SQL file
restaurant_data = []
pattern = r"INSERT INTO.*?VALUES\s*\((.*?)\);"
matches = re.findall(pattern, sql_content, re.DOTALL | re.IGNORECASE)

if not matches:
    print("ERROR: Could not parse SQL file!")
    print("First 500 characters:")
    print(sql_content[:500])
    exit()

for match in matches:
    # Split by comma and clean
    parts = [p.strip().strip("'").strip('"') for p in match.split(',')]
    restaurant_data.append(parts)

# Create restaurants DataFrame
# Based on what you said: columns are restaurant_id, name, cuisine, rating
restaurants_df = pd.DataFrame(restaurant_data, columns=['restaurant_id', 'name', 'cuisine', 'rating'])
restaurants_df['restaurant_id'] = restaurants_df['restaurant_id'].astype(str)

print(f"Loaded {len(restaurants_df)} restaurants")
print("\nSample restaurants with real cuisine names:")
print(restaurants_df[['restaurant_id', 'name', 'cuisine']].head(10))

print(f"\nUnique cuisine names found: {restaurants_df['cuisine'].unique()}")

# ============================================
# STEP 2: LOAD ORDERS AND MERGE WITH REAL CUISINE NAMES
# ============================================
print("\n" + "="*60)
print("LOADING ORDERS AND MERGING WITH REAL CUISINE DATA")
print("="*60)

# Load orders
orders_df = pd.read_csv('orders.csv')
orders_df['restaurant_id'] = orders_df['restaurant_id'].astype(str)
orders_df['user_id'] = orders_df['user_id'].astype(str)

print(f"Loaded {len(orders_df)} orders")

# Load users
with open('users.json', 'r') as f:
    users_df = pd.DataFrame(json.load(f))
users_df['user_id'] = users_df['user_id'].astype(str)

# Merge orders with restaurants (to get real cuisine names)
orders_with_cuisine = pd.merge(orders_df, restaurants_df[['restaurant_id', 'cuisine']], 
                                on='restaurant_id', how='left')

# Then merge with users
final_df = pd.merge(orders_with_cuisine, users_df, on='user_id', how='left')

print(f"\nFinal merged dataset shape: {final_df.shape}")
print(f"Orders with cuisine info: {final_df['cuisine'].notna().sum()}")

# ============================================
# STEP 3: FIND AMOUNT COLUMN
# ============================================
if 'total_amount' in final_df.columns:
    amount_col = 'total_amount'
elif 'order_value' in final_df.columns:
    amount_col = 'order_value'
else:
    amount_cols = [col for col in final_df.columns if 'amount' in col.lower() or 'value' in col.lower()]
    amount_col = amount_cols[0] if amount_cols else None

print(f"\nUsing '{amount_col}' as amount column")

# ============================================
# STEP 4: CHECK DATA
# ============================================
print("\n" + "="*60)
print("DATA CHECK")
print("="*60)

print(f"\nUnique membership types:")
print(final_df['membership'].unique())

print(f"\nUnique cuisine types:")
cuisines = final_df['cuisine'].unique()
print(cuisines[:20])  # Show first 20
print(f"Total unique cuisines: {len(cuisines)}")

# Check if we have the cuisines from the question
options_cuisines = ['Indian', 'Italian', 'Chinese']
print(f"\nLooking for question cuisines:")
for cuisine in options_cuisines:
    count = (final_df['cuisine'] == cuisine).sum()
    print(f"{cuisine}: {count} orders")

# ============================================
# STEP 5: CALCULATE REVENUE FOR EACH COMBINATION
# ============================================
print("\n" + "="*60)
print("CALCULATING REVENUE BY COMBINATION")
print("="*60)

# Define the combinations from the question
combinations = [
    ('Gold', 'Indian'),
    ('Gold', 'Italian'),
    ('Regular', 'Indian'),
    ('Regular', 'Chinese')
]

results = []
print("\nRevenue for each combination:")
for membership, cuisine in combinations:
    # Filter for this combination
    mask = (final_df['membership'] == membership) & (final_df['cuisine'] == cuisine)
    revenue = final_df.loc[mask, amount_col].sum()
    orders = mask.sum()
    
    results.append({
        'combination': f"{membership} + {cuisine} cuisine",
        'membership': membership,
        'cuisine': cuisine,
        'revenue': revenue,
        'orders': orders
    })
    
    print(f"\n{membership} + {cuisine} cuisine:")
    print(f"  Revenue: ₹{revenue:,.2f}")
    print(f"  Orders: {orders}")
    if orders > 0:
        avg = revenue / orders
        print(f"  Average order value: ₹{avg:.2f}")

# ============================================
# STEP 6: FIND THE HIGHEST REVENUE COMBINATION
# ============================================
print("\n" + "="*60)
print("FINDING HIGHEST REVENUE COMBINATION")
print("="*60)

# Convert to DataFrame and sort
results_df = pd.DataFrame(results)
results_df = results_df.sort_values('revenue', ascending=False)

print("\nAll combinations sorted by revenue:")
for i, row in results_df.iterrows():
    print(f"{row['combination']}: ₹{row['revenue']:,.2f} ({row['orders']} orders)")

# Get the highest
highest = results_df.iloc[0]

print(f"\n" + "="*60)
print(f"FINAL ANSWER: {highest['combination']}")
print(f"   Revenue: ₹{highest['revenue']:,.2f}")
print(f"   Orders: {highest['orders']}")
print("="*60)


Loading restaurants.sql to get real cuisine names...
Loaded 500 restaurants

Sample restaurants with real cuisine names:
  restaurant_id           name  cuisine
0             1   Restaurant_1  Chinese
1             2   Restaurant_2   Indian
2             3   Restaurant_3  Mexican
3             4   Restaurant_4  Chinese
4             5   Restaurant_5  Chinese
5             6   Restaurant_6  Chinese
6             7   Restaurant_7  Italian
7             8   Restaurant_8   Indian
8             9   Restaurant_9  Italian
9            10  Restaurant_10  Chinese

Unique cuisine names found: <StringArray>
['Chinese', 'Indian', 'Mexican', 'Italian']
Length: 4, dtype: str

LOADING ORDERS AND MERGING WITH REAL CUISINE DATA
Loaded 10000 orders

Final merged dataset shape: (10000, 10)
Orders with cuisine info: 10000

Using 'total_amount' as amount column

DATA CHECK

Unique membership types:
<StringArray>
['Regular', 'Gold']
Length: 2, dtype: str

Unique cuisine types:
<StringArray>
['Mexican', 'Ind

In [20]:
import pandas as pd
import json

# ============================================
# STEP 1: LOAD AND PREPARE DATA
# ============================================
print("Loading data...")

# Load orders
orders_df = pd.read_csv('orders.csv')

# Convert order_date to datetime
orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])

print(f"Loaded {len(orders_df)} orders")
print(f"Date range: {orders_df['order_date'].min().date()} to {orders_df['order_date'].max().date()}")

# ============================================
# STEP 2: FIND AMOUNT COLUMN
# ============================================
if 'total_amount' in orders_df.columns:
    amount_col = 'total_amount'
elif 'order_value' in orders_df.columns:
    amount_col = 'order_value'
else:
    amount_cols = [col for col in orders_df.columns if 'amount' in col.lower() or 'value' in col.lower()]
    amount_col = amount_cols[0] if amount_cols else None

print(f"Using '{amount_col}' as amount column")

# ============================================
# STEP 3: EXTRACT QUARTER FROM DATE
# ============================================
print("\n" + "="*60)
print("EXTRACTING QUARTER INFORMATION")
print("="*60)

# Extract quarter (Q1: Jan-Mar, Q2: Apr-Jun, Q3: Jul-Sep, Q4: Oct-Dec)
orders_df['quarter'] = orders_df['order_date'].dt.quarter

# Create a mapping for display
quarter_names = {
    1: 'Q1 (Jan–Mar)',
    2: 'Q2 (Apr–Jun)', 
    3: 'Q3 (Jul–Sep)',
    4: 'Q4 (Oct–Dec)'
}

orders_df['quarter_name'] = orders_df['quarter'].map(quarter_names)

# Also extract year for year-wise analysis
orders_df['year'] = orders_df['order_date'].dt.year

print(f"\nOrders by quarter:")
quarter_counts = orders_df['quarter_name'].value_counts().sort_index()
for quarter, count in quarter_counts.items():
    print(f"{quarter}: {count} orders")

# ============================================
# STEP 4: CALCULATE REVENUE BY QUARTER
# ============================================
print("\n" + "="*60)
print("CALCULATING REVENUE BY QUARTER")
print("="*60)

# Calculate total revenue by quarter
revenue_by_quarter = orders_df.groupby('quarter_name')[amount_col].sum().sort_index()

print("\nTotal revenue by quarter:")
for quarter_name, revenue in revenue_by_quarter.items():
    print(f"{quarter_name}: ₹{revenue:,.2f}")

# ============================================
# STEP 5: FIND THE QUARTER WITH HIGHEST REVENUE
# ============================================
print("\n" + "="*60)
print("FINDING QUARTER WITH HIGHEST REVENUE")
print("="*60)

# Find the quarter with maximum revenue
highest_quarter = revenue_by_quarter.idxmax()
highest_revenue = revenue_by_quarter.max()

print(f"\n📝 HIGHEST REVENUE QUARTER: {highest_quarter}")
print(f"   Revenue: ₹{highest_revenue:,.2f}")

# Show ranking
print(f"\nRanking of quarters by revenue:")
revenue_sorted = revenue_by_quarter.sort_values(ascending=False)
for i, (quarter, revenue) in enumerate(revenue_sorted.items(), 1):
    print(f"{i}. {quarter}: ₹{revenue:,.2f}")

# ============================================
# STEP 6: YEAR-WISE ANALYSIS (IF MULTIPLE YEARS)
# ============================================
print("\n" + "="*60)
print("YEAR-WISE ANALYSIS")
print("="*60)

# Check if we have multiple years
years = orders_df['year'].unique()
print(f"\nYears in data: {sorted(years)}")

if len(years) > 1:
    print("\nRevenue by quarter and year:")
    
    # Create pivot table
    pivot = orders_df.pivot_table(
        values=amount_col,
        index='year',
        columns='quarter_name',
        aggfunc='sum',
        fill_value=0
    )
    
    # Reorder columns to match Q1, Q2, Q3, Q4
    column_order = ['Q1 (Jan–Mar)', 'Q2 (Apr–Jun)', 'Q3 (Jul–Sep)', 'Q4 (Oct–Dec)']
    pivot = pivot.reindex(columns=[col for col in column_order if col in pivot.columns])
    
    print(pivot)
    
    # Find highest quarter for each year
    print("\nHighest revenue quarter for each year:")
    for year in sorted(years):
        year_data = orders_df[orders_df['year'] == year]
        year_revenue = year_data.groupby('quarter_name')[amount_col].sum()
        if not year_revenue.empty:
            highest = year_revenue.idxmax()
            revenue = year_revenue.max()
            print(f"{year}: {highest} (₹{revenue:,.2f})")
    
    # Overall highest across all years
    print(f"\n📝 OVERALL HIGHEST ACROSS ALL YEARS: {highest_quarter}")
else:
    print(f"All orders are from year: {years[0]}")

# ============================================
# STEP 7: ADDITIONAL ANALYSIS
# ============================================
print("\n" + "="*60)
print("ADDITIONAL ANALYSIS")
print("="*60)

# Monthly revenue for more granular view
print("\nMonthly revenue breakdown:")
orders_df['month'] = orders_df['order_date'].dt.month_name()
monthly_revenue = orders_df.groupby('month')[amount_col].sum()

# Order months chronologically
month_order = ['January', 'February', 'March', 'April', 'May', 'June',
               'July', 'August', 'September', 'October', 'November', 'December']
monthly_revenue = monthly_revenue.reindex([m for m in month_order if m in monthly_revenue.index])

for month, revenue in monthly_revenue.items():
    print(f"{month}: ₹{revenue:,.2f}")

# Percentage of total revenue by quarter
total_revenue = orders_df[amount_col].sum()
print(f"\nPercentage of total revenue by quarter:")
for quarter_name, revenue in revenue_by_quarter.items():
    percentage = (revenue / total_revenue) * 100
    print(f"{quarter_name}: {percentage:.1f}% (₹{revenue:,.2f})")

print(f"\nTotal annual revenue: ₹{total_revenue:,.2f}")



Loading data...
Loaded 10000 orders
Date range: 2023-01-01 to 2024-01-01
Using 'total_amount' as amount column

EXTRACTING QUARTER INFORMATION

Orders by quarter:
Q1 (Jan–Mar): 2519 orders
Q2 (Apr–Jun): 2440 orders
Q3 (Jul–Sep): 2522 orders
Q4 (Oct–Dec): 2519 orders

CALCULATING REVENUE BY QUARTER

Total revenue by quarter:
Q1 (Jan–Mar): ₹2,010,626.64
Q2 (Apr–Jun): ₹1,945,348.72
Q3 (Jul–Sep): ₹2,037,385.10
Q4 (Oct–Dec): ₹2,018,263.66

FINDING QUARTER WITH HIGHEST REVENUE

📝 HIGHEST REVENUE QUARTER: Q3 (Jul–Sep)
   Revenue: ₹2,037,385.10

Ranking of quarters by revenue:
1. Q3 (Jul–Sep): ₹2,037,385.10
2. Q4 (Oct–Dec): ₹2,018,263.66
3. Q1 (Jan–Mar): ₹2,010,626.64
4. Q2 (Apr–Jun): ₹1,945,348.72

YEAR-WISE ANALYSIS

Years in data: [np.int32(2023), np.int32(2024)]

Revenue by quarter and year:
quarter_name  Q1 (Jan–Mar)  Q2 (Apr–Jun)  Q3 (Jul–Sep)  Q4 (Oct–Dec)
year                                                                
2023            1993425.14    1945348.72     2037385.1    20182

/var/folders/6s/8th7nmh12sgfyhbp5sfjfjqw0000gn/T/ipykernel_48806/3373971297.py:13: UserWarning: Parsing dates in %d-%m-%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  orders_df['order_date'] = pd.to_datetime(orders_df['order_date'])


In [4]:
import pandas as pd

# ============================================
# STEP 1: LOAD AND MERGE DATA
# ============================================
print("Loading data...")

# Load orders
orders_df = pd.read_csv('orders.csv')

# Load users
with open('users.json', 'r') as f:
    import json
    users_df = pd.DataFrame(json.load(f))

# Convert IDs to string for merging
orders_df['user_id'] = orders_df['user_id'].astype(str)
users_df['user_id'] = users_df['user_id'].astype(str)

# Merge orders with users
merged_df = pd.merge(orders_df, users_df, on='user_id', how='left')

print(f"Total orders: {len(merged_df)}")

# ============================================
# STEP 2: IDENTIFY MEMBERSHIP COLUMN
# ============================================
print("\nIdentifying membership column...")

if 'membership' in merged_df.columns:
    membership_col = 'membership'
elif 'membership_type' in merged_df.columns:
    membership_col = 'membership_type'
else:
    # Find membership column
    membership_cols = [col for col in merged_df.columns if 'member' in col.lower()]
    if membership_cols:
        membership_col = membership_cols[0]
    else:
        print("ERROR: No membership column found!")
        print("Available columns:", merged_df.columns.tolist())
        exit()

print(f"Using '{membership_col}' as membership column")

# ============================================
# STEP 3: COUNT GOLD MEMBER ORDERS
# ============================================
print("\n" + "="*60)
print("COUNTING GOLD MEMBER ORDERS")
print("="*60)

# Check unique membership values
unique_memberships = merged_df[membership_col].unique()
print(f"Unique membership types: {unique_memberships}")

# Count orders by membership type
orders_by_membership = merged_df[membership_col].value_counts()
print(f"\nOrders by membership type:")
for membership, count in orders_by_membership.items():
    print(f"{membership}: {count} orders")

# Count Gold member orders
gold_orders = merged_df[merged_df[membership_col] == 'Gold']
gold_order_count = len(gold_orders)

print(f"\nGold member orders: {gold_order_count}")

# ============================================
# STEP 5: FINAL ANSWER
# ============================================
print("\n" + "="*60)
print("FINAL ANSWER")
print("="*60)
print(f"Total orders placed by Gold members: {gold_order_count}")
print("="*60)

Loading data...
Total orders: 10000

Identifying membership column...
Using 'membership' as membership column

COUNTING GOLD MEMBER ORDERS
Unique membership types: <StringArray>
['Regular', 'Gold']
Length: 2, dtype: str

Orders by membership type:
Regular: 5013 orders
Gold: 4987 orders

Gold member orders: 4987

FINAL ANSWER
Total orders placed by Gold members: 4987


In [9]:
import pandas as pd

# ============================================
# STEP 1: LOAD YOUR MERGED DATASET
# ============================================
print("Loading dataset...")
df = pd.read_csv('final_food_delivery_dataset.csv')

print(f"Dataset shape: {df.shape}")

# ============================================
# STEP 2: IDENTIFY COLUMNS
# ============================================
print("\nIdentifying columns...")

# Find city column
city_cols = [col for col in df.columns if 'city' in col.lower()]
if city_cols:
    city_col = city_cols[0]
    print(f"Using '{city_col}' as city column")
else:
    print("ERROR: No city column found!")
    print("Available columns:", df.columns.tolist())
    exit()

# Find amount column
if 'total_amount' in df.columns:
    amount_col = 'total_amount'
elif 'order_value' in df.columns:
    amount_col = 'order_value'
else:
    amount_cols = [col for col in df.columns if 'amount' in col.lower() or 'value' in col.lower()]
    if amount_cols:
        amount_col = amount_cols[0]
    else:
        print("ERROR: No amount column found!")
        print("Available columns:", df.columns.tolist())
        exit()

print(f"Using '{amount_col}' as amount column")

# ============================================
# STEP 3: CHECK CITY VALUES
# ============================================
print("\n" + "="*60)
print("CHECKING CITY DATA")
print("="*60)

print(f"\nUnique cities in data:")
unique_cities = df[city_col].unique()
for i, city in enumerate(sorted(unique_cities)[:20], 1):  # Show first 20
    print(f"{i}. {city}")

# Check for Hyderabad (case-insensitive)
hyderabad_variants = []
for city in unique_cities:
    if isinstance(city, str) and 'hyderabad' in city.lower():
        hyderabad_variants.append(city)

print(f"\nLooking for Hyderabad...")
if hyderabad_variants:
    print(f"Found Hyderabad variants: {hyderabad_variants}")
    # Use the most common variant
    hyderabad_city = hyderabad_variants[0]
else:
    print("⚠️ 'Hyderabad' not found in city names!")
    print("\nPossible matches (case-insensitive search):")
    for city in unique_cities:
        if isinstance(city, str):
            city_lower = city.lower()
            if any(keyword in city_lower for keyword in ['hyd', 'hyder']):
                print(f"  - {city} (possible match)")
    
    # If still not found, check for common abbreviations
    common_abbrev = {
        'HYD': 'Hyderabad',
        'Hyd': 'Hyderabad', 
        'Hyderabad City': 'Hyderabad'
    }
    
    for abbrev, full_name in common_abbrev.items():
        if abbrev in unique_cities:
            hyderabad_city = abbrev
            print(f"\nFound abbreviation: {abbrev} -> treating as Hyderabad")
            break
    else:
        print("\nNo Hyderabad found. Showing all cities with orders:")
        city_revenue = df.groupby(city_col)[amount_col].sum().sort_values(ascending=False)
        for city, revenue in city_revenue.head(10).items():
            print(f"{city}: ₹{revenue:,.2f}")
        exit()

# ============================================
# STEP 4: CALCULATE HYDERABAD REVENUE
# ============================================
print("\n" + "="*60)
print("CALCULATING HYDERABAD REVENUE")
print("="*60)

# Filter for Hyderabad orders
hyderabad_mask = df[city_col].astype(str).str.lower() == hyderabad_city.lower()
hyderabad_orders = df[hyderabad_mask]

print(f"Orders from {hyderabad_city}: {len(hyderabad_orders)}")

if len(hyderabad_orders) > 0:
    # Calculate total revenue
    total_revenue = hyderabad_orders[amount_col].sum()
    rounded_revenue = round(total_revenue)
    
    print(f"\nRevenue from {hyderabad_city}:")
    print(f"  Exact: ₹{total_revenue:,.2f}")
    print(f"  Rounded: ₹{rounded_revenue:,.0f}")
    
    
    # Percentage of total revenue
    total_all_revenue = df[amount_col].sum()
    percentage = (total_revenue / total_all_revenue) * 100
    print(f"  Percentage of total revenue: {percentage:.1f}%")
    
    # ============================================
    # STEP 5: FINAL ANSWER
    # ============================================
    print("\n" + "="*60)
    print("FINAL ANSWER")
    print("="*60)
    print(f"Total revenue from Hyderabad: ₹{rounded_revenue:,}")
    print(f"   (Rounded from ₹{total_revenue:,.2f})")
    print("="*60)
    
else:
    print(f"⚠️ No orders found from {hyderabad_city}!")



Loading dataset...
Dataset shape: (10000, 10)

Identifying columns...
Using 'city' as city column
Using 'total_amount' as amount column

CHECKING CITY DATA

Unique cities in data:
1. Bangalore
2. Chennai
3. Hyderabad
4. Pune

Looking for Hyderabad...
Found Hyderabad variants: ['Hyderabad']

CALCULATING HYDERABAD REVENUE
Orders from Hyderabad: 2350

Revenue from Hyderabad:
  Exact: ₹1,889,366.58
  Rounded: ₹1,889,367
  Percentage of total revenue: 23.6%

FINAL ANSWER
Total revenue from Hyderabad: ₹1,889,367
   (Rounded from ₹1,889,366.58)


In [11]:
import pandas as pd

# Load orders.csv
orders = pd.read_csv('orders.csv')

# Count distinct users
distinct_users = orders['user_id'].nunique()

print(f"ANSWER: {distinct_users}")
print(f"Total orders: {len(orders)}")

ANSWER: 2883
Total orders: 10000


In [13]:
#using orders and users seperately
import pandas as pd
import json

# Load and merge
orders = pd.read_csv('orders.csv')
with open('users.json', 'r') as f:
    users = pd.DataFrame(json.load(f))

merged = pd.merge(orders, users, on='user_id', how='left')

# Find columns
membership_col = 'membership' if 'membership' in merged.columns else [col for col in merged.columns if 'member' in col.lower()][0]
amount_col = 'total_amount' if 'total_amount' in merged.columns else 'order_value'

# Calculate
gold_avg = merged[merged[membership_col] == 'Gold'][amount_col].mean()
rounded_avg = round(gold_avg, 2)

print(f"ANSWER: ₹{rounded_avg:.2f}")

ANSWER: ₹797.15


In [15]:
import pandas as pd
import re

# Load and parse restaurants
with open('restaurants.sql', 'r') as f:
    sql = f.read()

# Parse SQL
data = []
for match in re.findall(r"INSERT INTO.*?VALUES\s*\((.*?)\);", sql, re.DOTALL | re.IGNORECASE):
    parts = [p.strip().strip("'").strip('"') for p in match.split(',')]
    data.append(parts)

restaurants = pd.DataFrame(data, columns=['id', 'name', 'cuisine', 'rating'])
restaurants['rating'] = pd.to_numeric(restaurants['rating'], errors='coerce')
restaurants['id'] = restaurants['id'].astype(str)

# Load orders
orders = pd.read_csv('orders.csv')
orders['restaurant_id'] = orders['restaurant_id'].astype(str)

# Merge
merged = pd.merge(orders, restaurants[['id', 'rating']], 
                  left_on='restaurant_id', right_on='id', how='left')

# Count orders with rating ≥ 4.5
count = len(merged[merged['rating'] >= 4.5])

print(f"ANSWER: {count}")
print(f"Total orders: {len(merged)}")
print(f"Percentage: {(count/len(merged))*100:.1f}%")

ANSWER: 3374
Total orders: 10000
Percentage: 33.7%


In [17]:
import pandas as pd

# Load data
df = pd.read_csv('final_food_delivery_dataset.csv')

# Find columns
membership_col = 'membership' if 'membership' in df.columns else [col for col in df.columns if 'member' in col.lower()][0]
city_col = [col for col in df.columns if 'city' in col.lower()][0]
amount_col = 'total_amount' if 'total_amount' in df.columns else 'order_value'

# Filter Gold members
gold_df = df[df[membership_col] == 'Gold']

# Find top revenue city for Gold members
top_city = gold_df.groupby(city_col)[amount_col].sum().idxmax()

# Count orders in that city
orders_count = len(gold_df[gold_df[city_col] == top_city])

print(f"ANSWER: {orders_count}")
print(f"Top revenue city for Gold members: {top_city}")
print(f"Gold orders in {top_city}: {orders_count}")

ANSWER: 1337
Top revenue city for Gold members: Chennai
Gold orders in Chennai: 1337
